# Classification des œuvres de Zola et d'autres naturalistes

## Expérience avec retrait d'une liste de noms propres

## 1. Importation des bibliothèques

In [1]:
# Fichiers et données
import gc
import glob
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Représentation textuelle
from sklearn.feature_extraction.text import TfidfVectorizer

# Modèles étudiés
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC

# Pipeline de la baseline « artefacts seulement »
from sklearn.preprocessing import StandardScaler

# Validation groupée
from sklearn.model_selection import StratifiedGroupKFold

# Évaluation
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

GRAINE_ALEATOIRE = 42
random.seed(GRAINE_ALEATOIRE)
np.random.seed(GRAINE_ALEATOIRE)

print("Toutes les bibliothèques sont chargées.")


Toutes les bibliothèques sont chargées.


## 2. Normalisation des apostrophes

Les textes ayant été préalablement nettoyés, certaines apostrophes ont
disparu. Cette étape restaure les contractions les plus fréquentes à partir
d'un dictionnaire de correspondances.

N.b: Les formes `dune` et `lune` sont respectivement interprétées comme `d'une`
et `l'une`. Ce choix repose sur l'hypothèse que les occurrences des noms
communs *dune* et *lune* sont minoritaires dans le corpus étudié.

In [2]:
REMPLACEMENTS_APOSTROPHES = {
    # ========================================================
    # Locutions longues : à traiter avant les formes courtes
    # ========================================================
    "lorsquil": "lorsqu'il",
    "lorsquils": "lorsqu'ils",
    "lorsquon": "lorsqu'on",
    "lorsquune": "lorsqu'une",
    "lorsquun": "lorsqu'un",
    "lorsquelle": "lorsqu'elle",
    "lorsquelles": "lorsqu'elles",

    "puisquil": "puisqu'il",
    "puisquils": "puisqu'ils",
    "puisquon": "puisqu'on",
    "puisquune": "puisqu'une",
    "puisquun": "puisqu'un",
    "puisquelle": "puisqu'elle",
    "puisquelles": "puisqu'elles",

    "quoiquil": "quoiqu'il",
    "quoiquils": "quoiqu'ils",
    "quoiquon": "quoiqu'on",
    "quoiquelle": "quoiqu'elle",

    "jusquil": "jusqu'il",
    "jusquils": "jusqu'ils",
    "jusquon": "jusqu'on",
    "jusquune": "jusqu'une",
    "jusquun": "jusqu'un",
    "jusquelle": "jusqu'elle",
    "jusquelles": "jusqu'elles",
    "jusquà": "jusqu'à",
    "jusquau": "jusqu'au",
    "jusquaux": "jusqu'aux",

    # ========================================================
    # Contractions avec c'
    # ========================================================
    "cest": "c'est",
    "cétait": "c'était",
    "cétaient": "c'étaient",
    "cétant": "c'étant",
    "cen": "c'en",

    # ========================================================
    # Contractions avec s'
    # ========================================================
    "sest": "s'est",
    "sétait": "s'était",
    "sétaient": "s'étaient",
    "sétant": "s'étant",
    "sen": "s'en",
    "sy": "s'y",
    "sil": "s'il",
    "sils": "s'ils",

    # ========================================================
    # Contractions avec n'
    # ========================================================
    "nest": "n'est",
    "nétait": "n'était",
    "nétaient": "n'étaient",
    "navait": "n'avait",
    "navaient": "n'avaient",
    "naurait": "n'aurait",
    "nauraient": "n'auraient",
    "naura": "n'aura",
    "nauront": "n'auront",
    "na": "n'a",
    "nont": "n'ont",
    "nen": "n'en",
    "ny": "n'y",
    "navoir": "n'avoir",
    "nêtre": "n'être",
    "nai": "n'ai",
    "navez": "n'avez",

    # ========================================================
    # Contractions avec j'
    # ========================================================
    "jai": "j'ai",
    "javais": "j'avais",
    "jétais": "j'étais",
    "jaurais": "j'aurais",
    "jaurai": "j'aurai",
    "jaime": "j'aime",
    "jallais": "j'allais",
    "jignore": "j'ignore",
    "jentends": "j'entends",
    "jen": "j'en",
    "jy": "j'y",
    "jirai": "j'irai",
    "tai": "t'ai",

    # ========================================================
    # Contractions avec qu'
    # ========================================================
    "quest": "qu'est",
    "quil": "qu'il",
    "quils": "qu'ils",
    "quon": "qu'on",
    "quun": "qu'un",
    "quune": "qu'une",
    "quà": "qu'à",
    "quen": "qu'en",


    # ========================================================
    # Contractions avec d'
    # ========================================================
    "dune": "d'une", #je prend le parti de mettre d'apostrophe mais il y a un risque de confusion avec "d'une" (de une) et "dune" (la dune) 
    "dun": "d'un",
    "delle": "d'elle",
    "delles": "d'elles",
    "dêtre": "d'être",
    "davoir": "d'avoir",
    "dabord": "d'abord",
    "dailleurs": "d'ailleurs",
    "daprès": "d'après",
    "daccord": "d'accord",
    "davance": "d'avance",
    "doù": "d'où",
    "dautres": "d'autres",
    "damour": "d'amour",
    "dargent": "d'argent",
    "despoir": "d'espoir",
    "den": "d'en",
    "desprit": "d'esprit",
    'deau': "d'eau",

    # ========================================================
    # Contractions avec l'
    # ========================================================
    "labbé": "l'abbé",
    "lair": "l'air",
    "lâme": "l'âme",
    "lami": "l'ami",
    "lamie": "l'amie",
    "lamour": "l'amour",
    "largent": "l'argent",
    "lautre": "l'autre",
    "lun": "l'un",
    "lune": "l'une",
    "lheure": "l'heure",
    "lhomme": "l'homme",
    "lhonneur": "l'honneur",
    "lhistoire": "l'histoire",
    "lhôtel": "l'hôtel",
    "léglise": "l'église",
    "lépoque": "l'époque",
    "lenfant": "l'enfant",
    "lendroit": "l'endroit",
    "lentrée": "l'entrée",
    "lintérieur": "l'intérieur",
    "lidée": "l'idée",
    "lombre": "l'ombre",
    "loeil": "l'œil",
    "lœil": "l'œil",
    "lon": "l'on",
    "lai": "l'ai",
    "lavait": "l'avait",
    "leau": "l'eau",
    "lesprit": "l'esprit",
    "lescalier": "l'escalier",
    "létudiant": "l'étudiant",
    'lavenir': "l'avenir",
    "laffaire": "l'affaire",
    "loreille": "l'oreille",
    "lavocat": "l'avocat",

    # ========================================================
    # Locutions courantes
    # ========================================================
    "aujourdhui": "aujourd'hui",
    "presquun": "presqu'un",
    "presquune": "presqu'une",
    "quelquun": "quelqu'un",
    "quelquune": "quelqu'une",
    "quaprès": "qu'après",

    # ========================================================
    # Verbes pronominaux fréquemment rencontrés
    # ========================================================
    "sécria": "s'écria",
    "sécriait": "s'écriait",
    "sécrièrent": "s'écrièrent",
    "saperçut": "s'aperçut",
    "sapercevait": "s'apercevait",
    "sapprocha": "s'approcha",
    "sapprochait": "s'approchait",
    "sarrêta": "s'arrêta",
    "sarrêtait": "s'arrêtait",
    "sassit": "s'assit",
    "sétendit": "s'étendit",
    "séloigna": "s'éloigna",
    "séloignait": "s'éloignait",
    "sêtre" : "s'être",
    
    # m'
    "men": "m'en",
    "mavez": "m'avez",
    "mavait": "m'avait",
    
    "daller": "d'aller",
    "lappartement": "l'appartement",
    "lhuile": "l'huile",
    "mest": "m'est",
    "dhonneur": "d'honneur",
    "dici": "d'ici",
    "neût": "n'eût",
    "sécrie": "s'écrie",
    "ce quelle": "ce qu'elle",
    "ten": "t'en",
    "dhenriette": "d'Henriette",
    "lart": "l'art",
    "quau": "qu'au",
    "quaux": "qu'aux",
    "dautre": "d'autre",
    "dautres": "d'autres",
    "navais": "n'avais",
    "navions": "n'avions",
    "sagit": "s'agit",
    
    "dœil": "d'œil", 
    "létat": "l'état", 
    "leffet": "l'effet", 
    "jespère": "j'espère", 
    "lhabitude": "l'habitude", 
    "nêtes": "n'êtes",
    "dy": "d'y",
    "linstant": "l'instant",
    "nimporte": "n'importe",
    
    
    #=============================
    #Artecfact OCR
    #=============================
    "jé":"je",
    "dé":"de", 
    "lé": "le", 
    "mé":"me", 
    "cé":"ce", 
    "qué":"que",
    "dor": "d'or", 
    "my": "m'y", 
    "ten": "t'en",
    "daffaires": "d'affaires",
    "daimer": "d'aimer",
    "dair": "d'air",
    "dangoisse": "d'angoisse",
    "dannées": "d'années",
    "darbres": "d'arbres",
    "darmes": "d'armes",
    "darriver": "d'arriver",
    "dart": "d'art",
    "dattendre": "d'attendre",
    "dautant": "d'autant",
    "dautrefois": "d'autrefois",
    "denfant": "d'enfant",
    "denfants": "d'enfants",
    "dentendre": "d'entendre",
    "dentrer": "d'entrer",
    "dhabitude": "d'habitude",
    "dheure": "d'heure",
    "dhomme": "d'homme",
    "dhommes": "d'hommes",
    "didées": "d'idées",
    "dimpatience": "d'impatience",
    "dinquiétude": "d'inquiétude",
    "douvrir": "d'ouvrir",
    "dénormes": "d'énormes",
    "dété": "d'été",
    "dœuvre": "d'œuvre",
    "jaimerais": "j'aimerais",
    "labri": "l'abri",
    "lacte": "l'acte",
    "laction": "l'action",
    "ladresse": "l'adresse",
    "laimait": "l'aimait",
    "laime": "l'aime",
    "laimer": "l'aimer",
    "laise": "l'aise",
    "lallée": "l'allée",
    "lamant": "l'amant",
    "lamitié": "l'amitié",
    "lancien": "l'ancien",
    "lancienne": "l'ancienne",
    "lannée": "l'année",
    "lantichambre": "l'antichambre",
    "lappelait": "l'appelait",
    "laprès": "l'après",
    "larrivée": "l'arrivée",
    "larrière": "l'arrière",
    "larrêta": "l'arrêta",
    "lartiste": "l'artiste",
    "laspect": "l'aspect",
    "lattaque": "l'attaque",
    "lattendait": "l'attendait",
    "lattente": "l'attente",
    "lattention": "l'attention",
    "lattitude": "l'attitude",
    "lauberge": "l'auberge",
    "laurais": "l'aurais",
    "laurait": "l'aurait",
    "lauteur": "l'auteur",
    "laventure": "l'aventure",
    "layant": "l'ayant",
    "lembarras": "l'embarras",
    "lembrassa": "l'embrassa",
    "lembrasser": "l'embrasser",
    "lennemi": "l'ennemi",
    "lennui": "l'ennui",
    "lentendre": "l'entendre",
    "lentretien": "l'entretien",
    "lenvie": "l'envie",
    "lespace": "l'espace",
    "lespoir": "l'espoir",
    "lespérance": "l'espérance",
    "lestomac": "l'estomac",
    "lexistence": "l'existence",
    "lexpression": "l'expression",
    "lexpérience": "l'expérience",
    "leût": "l'eût",
    "lherbe": "l'herbe",
    "lhiver": "l'hiver",
    "lhorizon": "l'horizon",
    "lhorreur": "l'horreur",
    "lhumanité": "l'humanité",
    "limage": "l'image",
    "limagination": "l'imagination",
    "limmense": "l'immense",
    "limpression": "l'impression",
    "linconnu": "l'inconnu",
    "linfluence": "l'influence",
    "linquiétude": "l'inquiétude",
    "linstinct": "l'instinct",
    "lintelligence": "l'intelligence",
    "linterrompit": "l'interrompit",
    "lintérêt": "l'intérêt",
    "lobscurité": "l'obscurité",
    "loccasion": "l'occasion",
    "lont": "l'ont",
    "lopinion": "l'opinion",
    "lopéra": "l'opéra",
    "lor": "l'or",
    "lorage": "l'orage",
    "lordre": "l'ordre",
    "louvrage": "l'ouvrage",
    "lusage": "l'usage",
    "ly": "l'y",
    "lâge": "l'âge",
    "léclat": "l'éclat",
    "lécole": "l'école",
    "lécoutait": "l'écoutait",
    "léducation": "l'éducation",
    "lémotion": "l'émotion",
    "lépaule": "l'épaule",
    "létoffe": "l'étoffe",
    "létranger": "l'étranger",
    "létude": "l'étude",
    "lété": "l'été",
    "lêtre": "l'être",
    "lœuvre": "l'œuvre",
    "maime": "m'aime",
    "naimait": "n'aimait",
    "naime": "n'aime",
    "naurais": "n'aurais",
    "navons": "n'avons",
    "nayant": "n'ayant",
    "nentendait": "n'entendait",
    "neut": "n'eut",
    "nosa": "n'osa",
    "nosait": "n'osait",
    "nosant": "n'osant",
    "nétais": "n'étais",
    "nétant": "n'étant",
    "quavait": "qu'avait",
    "quavec": "qu'avec",
    "quy": "qu'y",
    "sadressant": "s'adressant",
    "sagissait": "s'agissait",
    "samusait": "s'amusait",
    "sarrêter": "s'arrêter",
    "sasseoir": "s'asseoir",
    "sattendait": "s'attendait",
    "savança": "s'avança",
    "savançait": "s'avançait",
    "sefforçait": "s'efforçait",
    "sempêcher": "s'empêcher",
    "sexpliquer": "s'expliquer",
    "soccupait": "s'occupait",
    "soccuper": "s'occuper",
    "souvrit": "s'ouvrit",
    "sélança": "s'élança",
    "séleva": "s'éleva",
    "sélevait": "s'élevait",
    "séloigner": "s'éloigner",
    "sétendait": "s'étendait",
    "sétonnait": "s'étonnait",
    "taime": "t'aime",
}

# Ces formes ont plusieurs analyses possibles et ne doivent pas
# être corrigées globalement sans examiner leur contexte.
FORMES_AMBIGUES_NON_CORRIGEES = {"lune", "dune", "dé", "qué"}
for forme_ambigue in FORMES_AMBIGUES_NON_CORRIGEES:
    REMPLACEMENTS_APOSTROPHES.pop(forme_ambigue, None)

def adapter_casse(forme_originale, remplacement):
    """Conserve approximativement la casse du mot d'origine."""

    if forme_originale.isupper():
        return remplacement.upper()

    if forme_originale[0].isupper():
        return remplacement[0].upper() + remplacement[1:]

    return remplacement


FORMES_APOSTROPHES_TRIEES = sorted(
    REMPLACEMENTS_APOSTROPHES,
    key=len,
    reverse=True
)

MOTIF_APOSTROPHES = re.compile(
    r"\b(?:"
    + "|".join(
        re.escape(forme)
        for forme in FORMES_APOSTROPHES_TRIEES
    )
    + r")\b",
    flags=re.IGNORECASE
)


def restaurer_apostrophes(texte):
    """Restaure en une passe les apostrophes à forte confiance."""

    def remplacer(correspondance):
        forme_originale = correspondance.group(0)
        forme_corrigee = REMPLACEMENTS_APOSTROPHES[
            forme_originale.lower()
        ]
        return adapter_casse(
            forme_originale,
            forme_corrigee
        )

    return MOTIF_APOSTROPHES.sub(remplacer, texte)

## 3. Fenêtres fixes et jeu de développement

Chaque œuvre est découpée en fenêtres non chevauchantes de 500 tokens.
Le reliquat incomplet est écarté afin que la longueur ne permette pas au
modèle de distinguer les classes. Le texte brut est conservé dans les
fenêtres : les cinq variantes seront dérivées ensuite des mêmes lignes.

L'ancien « test » devient explicitement un **jeu de développement**. Il ne
constitue plus une évaluation finale. Le découpage est groupé par œuvre.


In [3]:
TAILLE_FENETRE = 500
MOTIF_TOKEN_FENETRE = re.compile(r"(?u)\b\w+\b")


def segmenter_en_fenetres_fixes(texte, taille_fenetre=TAILLE_FENETRE):
    """Retourne des fenêtres de taille strictement identique en tokens."""

    numero = 0
    compteur = 0
    debut = None

    for position in MOTIF_TOKEN_FENETRE.finditer(texte):
        if compteur == 0:
            debut = position.start()

        compteur += 1
        if compteur == taille_fenetre:
            yield numero, texte[debut:position.end()]
            numero += 1
            compteur = 0


def determiner_label(nom_fichier):
    """Étiquetage provisoire hérité du notebook initial."""

    if nom_fichier.startswith("Émile_Zola"):
        return "Zola"
    return "naturaliste"


def construire_dataframe(liste_fichiers):
    donnees = []

    for chemin_fichier in liste_fichiers:
        nom_fichier = os.path.basename(chemin_fichier)
        label = determiner_label(nom_fichier)

        with open(chemin_fichier, "r", encoding="utf-8") as fichier:
            texte_brut = fichier.read()

        for numero, fenetre in segmenter_en_fenetres_fixes(texte_brut):
            donnees.append({
                "texte_brut": fenetre,
                "label": label,
                "source": nom_fichier,
                "fenetre": numero,
                "nombre_tokens": TAILLE_FENETRE,
            })

    return pd.DataFrame(donnees)


chemin_dossier = "Data_ZN"
liste_fichiers = sorted(
    glob.glob(os.path.join(chemin_dossier, "*.txt"))
)

if not liste_fichiers:
    raise FileNotFoundError(
        f"Aucun fichier .txt trouvé dans {Path(chemin_dossier).resolve()}"
    )

df_complet = construire_dataframe(liste_fichiers)

# Même logique que l'ancien hold-out, mais il est désormais nommé
# développement. Aucun test final n'est ouvert dans ce notebook.
separation_developpement = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=GRAINE_ALEATOIRE,
)

indices_train, indices_developpement = next(
    separation_developpement.split(
        X=df_complet["texte_brut"],
        y=df_complet["label"],
        groups=df_complet["source"],
    )
)

df_train = df_complet.iloc[indices_train].reset_index(drop=True)
df_developpement = (
    df_complet.iloc[indices_developpement].reset_index(drop=True)
)

print(f"Nombre de fichiers : {len(liste_fichiers)}")
print(f"Nombre total de fenêtres : {len(df_complet)}")
print(f"Taille de chaque fenêtre : {TAILLE_FENETRE} tokens")
print(f"Fenêtres d'apprentissage : {len(df_train)}")
print(f"Fenêtres de développement : {len(df_developpement)}")


Nombre de fichiers : 61
Nombre total de fenêtres : 11657
Taille de chaque fenêtre : 500 tokens
Fenêtres d'apprentissage : 9371
Fenêtres de développement : 2286


## 4. Vérification de la composition des corpus

Le tableau ci-dessous vérifie la séparation stricte des œuvres entre
l'apprentissage et le développement. Les variantes de texte ne changent
ni ces lignes, ni ces groupes.


In [4]:
def afficher_composition(nom, dataframe):
    print(f"\n{nom}")
    print("Fenêtres :", len(dataframe))
    print("Œuvres :", dataframe["source"].nunique())
    print(dataframe.groupby("label")["source"].nunique())


afficher_composition("APPRENTISSAGE", df_train)
afficher_composition("DÉVELOPPEMENT", df_developpement)

oeuvres_train = set(df_train["source"])
oeuvres_developpement = set(df_developpement["source"])
intersection = oeuvres_train & oeuvres_developpement

print("\nŒuvres communes :", len(intersection))
assert not intersection, "Une œuvre apparaît dans les deux ensembles."
assert df_complet["nombre_tokens"].eq(TAILLE_FENETRE).all()



APPRENTISSAGE
Fenêtres : 9371
Œuvres : 49
label
Zola           16
naturaliste    33
Name: source, dtype: int64

DÉVELOPPEMENT
Fenêtres : 2286
Œuvres : 12
label
Zola           4
naturaliste    8
Name: source, dtype: int64

Œuvres communes : 0


In [5]:
chemin_json = Path("patronime.json")

stop_words_noms_propres = [
"buteau", "florent",  "faujas", "pauline",  "coupeau", "plassans", "nana", "chanteau", "josserand", "miette", "lazare", "frédéric", "emma", "arnoux","faustin", "germinie", "duroy", "birotteau", "berthe","martine", "véronique", "kahn", "joachim", "mathéus", "louveau", "durtal", "émile", "forestier", "allart", "césar", "julien", "quesnoy", "francis", "dambreuse", "walter","deslauriers", "christiane", "gontran", "rosanette", "andermatt", "lorilleux", "maheude", "trublot",  "chaval", "delaherche", "fagerolles", "campardon", "delestang", "gavard", "poizat", "rastoil", "goujet", "condamin", "aurélie", "hubertine", "jeanlin", "bourdoncle", "gilquin", "mahoudeau", "robineau","jory", "hutin", "saturnin", "bouchard", "lerat", "chaudoreille", "urbain", "popinot", "corbie", "touquet","gorenflot", "sylvius","charlotte", "stauernaghel", "tillet", "julia", "césarine", "max","oriol","rosalie","hamilcar", "colombine", "roy", "montsou", "rasseneur", "bécu", "zacharie", "hubert", "trouche", "saget", "rognes",  "hourdequin", "chouteau", "charbonnel", "granoux", "gundermann", "gueulin", "marjolin", "boves", "loubet", "ramond","voreux", "gagnière", "marsy", "macqueron", "bird", "roguin", "mâtho","esseintes", "jenkins","spendius","homais", "eudeline", "baudouin", "paule", "popeland", "ragon", "niflart", "perrin", "izoard", "brétigny","rodolphe", "marcel", "anselme", "rouquette", "lapoulle", "dubuche", "gourd", "sarriette", "rochas", "mazaud", "bonneville", "vabre", "favier", "bazeilles", "paloque", "lengaigne", "béjuin","dide", "meuse", "henriette", "claparon", "jansoulet", "bigle", "tournelles", "lestang", "guay", "lefèvre", "carthage",  "sénécal","hussonnet", "germain", "félix", "juzeur", "delangre", "lecœur", "bongrand", "nénesse", "bonnemort", "bourrette", "vallagnosc", "jantrou", "deloche", "roudier", "neuville", "villebelle", "dina", "christian", "marelle","albine", "fauchery", "bordenave", "muffat", "lévise", "dingo", "andré", "cyprien", "simonne"    
]

#======================================================================================================================
#detection des noms en plus avec chatgpt Sol 5.6
# Formes supplémentaires repérées automatiquement dans le corpus.
NOMS_PROPRES_DETECTES_CORPUS = """
georgette marguerite risler christ stéphanie roubaud jules gianni pillerault lafleur frantz lacaille
sabine chèbe bélisaire zizine adèle célestine blossard jupillon athénaïs roudic bertin jenneval
clara jourdan rambaud steiner dolbert napoléon william annette félicia lison maria gaston
marcillon mâdou frédérique planus désableau demailly fanny delaberge laroche lucien misard pellerin
rosen mélie zénaïde cisy géry ildegonde maugendre piscot awah ida tonin régeant
gilberte hambert javel cabuche monpavon jaulin jouve lévis guilleroy malignon paulin archangias
moreau larsonneau martinon hirsch justin bourras huret roque cazenove xavier augustin noémie
pépé vatnaz bernard gallois barancy jacqueline pierron vuillet hannon lavertujeon théodore touanhô
honorat lebigre merville devaux gaga gaudissart artaud godard méchain robert wilkie beaudoin
bruno caquirol claudius baucharmais denizet maffre souvarine colomban fétu herbert pache balzac
félicie vauquelin achille agnès bébert péqueur martineau baudry bourdeu deburau prullière weber
minouche noël voltaire pichon saffré albert dambreville mauduit picard edgar keller mathilde
charvet lebas siméon delphin lanlaire lequeu madinier rosambeau toutin alcide bagot bouroche
célestin zizi guibal mariani baillehache benoît rolande bourdelais charrigaud chouard dejoie desgraves
michelin sauvaire anne bazaine blanmignon champfleury dansaert dobson eustache fauconnier marquès rouault
élisabeth binet bonnefille lebeau louisette norbert foucarmont gertrude gigonnet giscon juillerat leemans
vaudrec bambousse blanchart lebleu lénore merquier morainville pouillaud adalgis boscovich delmar grindot
moser sédille bescapé boisroger casta daubray guiraud horteur jacoby jacquand lefrançois maloir rudemar
sandorff billardière bourniche grandville archambauld banban bonaparte decostère douay gaudron guyons lamperière
molière mouquet adolphe durand gannec gouraud hillegrin ovide vuillaume bouland bouteloup doulinet
ducrot gabrielle lagniaud liénard naudet ruys tuvache castagnozoff chantemesse corbin cornille emile
flory gaujean maillobert mangin marcelle paulhat chédeville laurent maranne mary massias mazelli
ravenel bischoff cantel courbet crottat daignes hartmann matifat mussy villiers baal bron
clovis guénegaud malterre mariolle oudry tatan urtuby amanda axel baudelaire bournisien canivet
compan devreuse dumont gévresin loraux peirotte rousseau rozan rusconi thiers alexandrine bougon
chorche kimberly montbouillant ophélie rousselot sauvadon sophia bodin delacroix gobseck hippo néné
rochefontaine silvis soulas zaryte barca bonnehon dabadie delarocque derville durieu irène jeannine
joncquoy kolb léonide mardonnet pantois abel antonia bertaux boisrenard bourget crenmitz farandal
hugo lecomte léandre montceaux nathansohn orchère satan saïd taboureau verdier wimpffen afchin
barillot bompain caffin chabe chantereau delphine joliton lamartine madou mézières philozèle radicet
tony alphée camy carnavant cauche dalichamp doublet lamotte lauwerens léopold picot ragache
timbry zéphirine édouard balthazar cossart ducat grignoux jobelin krettly laurine léonce malindre
mézécourt valerio wagner zéphir bouchereau brûlard herpett isidore lamare lestiboudois minerve vulcain
éléonore alphonse billecoq boutarel didine foy gartlauben gaude gautier grandguillot laveau ozil
poe richomme rébufat zarxas beethoven camusot finet firmin garguille giroust gonin hezeta
landa lydwine marescot montenlair müller rengade salmon schopenhauer valentin violaine wattelet cassandre
cornât couillard dupré edward gallet gouin géraldine leforgeur manon mazel riquier selwyn
vanzade vinçart amadieu babet béranger chave compain diderot estrelle juan julio justine
lapotte lariboisière lastique lina lucette mabille pastil plinplan polge rochefoucauld ruysbroeck sauvagnat
shakespeare thuvin thésée tolbiac will alexis amaury angéline botticellina chermette corot dupont
euphorbe feutry gambetta george guillaumin horn lebrun madeline margueritte mauger meinhold musset
octavie pinggleton proudhon raguet rembrandt rolet salomé titien virgile voriau aillaume amy
andoche badinguet bossuet boutin brunet dante dickens dittmer emmerich euphémie farjasse fifine
fitz flaubert gabet godin guichon guizot jacob joé langlois lauréal lebescam ledru
luart mallarmé marcadet mozart pickersgill rollin rubens sand tissot vandenesse vanska
""".split()
#======================================================================================================================

mots_grammaticaux_a_conserver = {
    "de", "du", "des",
    "la", "le", "les",
    "d", "l"
}

# Même motif que le TfidfVectorizer par défaut : les noms composés
# et apostrophés sont découpés comme ils le seront dans le modèle.
MOTIF_TOKEN_TFIDF = re.compile(r"(?u)\b\w\w+\b")


def tokeniser_noms_propres(valeur):
    return {token.lower() 
            for token in MOTIF_TOKEN_TFIDF.findall(str(valeur or ""))}


def charger_noms_propres(chemin, noms_complementaires):
    with chemin.open("r", encoding="utf-8") as fichier:
        personnages = json.load(fichier)
    if not isinstance(personnages, list):
        raise ValueError("patronime.json doit contenir une liste de personnages.")

    noms_json = set()
    for indice, personnage in enumerate(personnages):
        if not isinstance(personnage, dict):
            raise ValueError(
                f"Entrée {indice} de patronime.json invalide : dictionnaire attendu."
            )
        for champ in ("prenom", "nom"):
            noms_json.update(
                tokeniser_noms_propres(personnage.get(champ, ""))
            )

    noms_ajoutes = set()
    for valeur in noms_complementaires:
        noms_ajoutes.update(tokeniser_noms_propres(valeur))

    noms_a_retirer = (noms_json | noms_ajoutes) - mots_grammaticaux_a_conserver

    return personnages, noms_json, noms_ajoutes, sorted(noms_a_retirer)

personnages_zola, noms_propres_json, noms_propres_ajoutes, stop_words = (
    charger_noms_propres(chemin_json,[*stop_words_noms_propres, *NOMS_PROPRES_DETECTES_CORPUS]))

stop_words_noms_propres_set = set(stop_words)

print("Entrées du JSON :", len(personnages_zola))
print("Tokens uniques issus du JSON :", len(noms_propres_json))
print("Tokens complémentaires uniques :", len(noms_propres_ajoutes))
print("Nombre total de noms propres retirés :", len(stop_words))
print("Mots grammaticaux conservés :", sorted(mots_grammaticaux_a_conserver))

# Les TF-IDF seront ajustés à l'intérieur de chaque pli.
# Aucun vocabulaire n'est appris globalement à ce stade.


Entrées du JSON : 240
Tokens uniques issus du JSON : 325
Tokens complémentaires uniques : 718
Nombre total de noms propres retirés : 1037
Mots grammaticaux conservés : ['d', 'de', 'des', 'du', 'l', 'la', 'le', 'les']


## 6. Cinq variantes construites sur les mêmes fenêtres

Les transformations sont déterministes et ne consultent jamais les
étiquettes. Elles sont appliquées après le découpage, ce qui garantit une
correspondance exacte entre les lignes des cinq variantes :

1. **texte brut** : fenêtre originale, sans normalisation ;
2. **filtrage manuel** : apostrophes restaurées et liste manuelle de noms
   retirée par le vectoriseur ;
3. **entités masquées** : entités du lexique contrôlé remplacées par un
   marqueur, complétées par les suites de mots capitalisés ;
4. **mots fonctionnels seuls** : seuls les déterminants, pronoms,
   prépositions, conjonctions, auxiliaires et négations sont conservés ;
5. **contenu lexical lemmatisé** : mots fonctionnels et entités retirés,
   puis lemmatisation des types lexicaux avec le modèle français spaCy.

La lemmatisation est calculée une fois par forme unique, puis gardée en
mémoire. Cela réduit fortement le temps d'exécution sans apprendre quoi que
ce soit à partir des classes.


In [6]:
MOTIF_TOKEN_VARIANTE = re.compile(r"(?u)\b\w+\b")

MOTS_FONCTIONNELS_FR = set("""
à afin ainsi alors après au aucun aucune aucuns aucunes auquel auxquels
aux auxquelles avec avant car ce ceci cela celle celles celui ceux chaque
chez comme comment dans de dedans dehors depuis des dès dessous dessus
devant donc dont du durant elle elles en entre et eux hors ici il ils je
jusque la laquelle lesquelles lequel lesquels le les leur leurs lui ma mais
me mes mien mienne miennes miens moi moins mon ne ni nos notre nôtre nôtres
nous on ou où par parce pendant personne peu plus pour pourquoi quand que
quel quelle quelles quels qui quoi quoique sa sans se selon ses si soi son
sous sur ta te tel telle telles tels tes toi ton tous tout toute toutes tu
un une vers voici voilà vos votre vôtre vôtres vous y
ai aie aient aies ait as avons avez ont avais avait avions aviez avaient
aurai auras aura aurons aurez auront aurais aurait aurions auriez auraient
eus eut eûmes eûtes eurent eu eue eues été étant être suis es est sommes
êtes sont étais était étions étiez étaient serai seras sera serons serez
seront serais serait serions seriez seraient sois soit soyons soyez soient
fus fut fûmes fûtes furent
cependant néanmoins pourtant puisque lorsque tandis sinon encore déjà
jamais toujours très trop aussi même mêmes seulement
""".split())

MOTIF_NOMS_PROPRES = re.compile(
    r"(?iu)\b(?:"
    + "|".join(
        re.escape(nom)
        for nom in sorted(
            stop_words_noms_propres_set,
            key=len,
            reverse=True,
        )
    )
    + r")\b"
)

# Une suite de deux formes capitalisées est traitée comme une entité
# probable. Les noms isolés sont couverts par le lexique contrôlé.
MOTIF_SUITE_CAPITALISEE = re.compile(
    r"(?u)\b(?:[A-ZÀ-ÖØ-Þ][a-zà-öø-ÿœæ]{2,})"
    r"(?:[\s-]+[A-ZÀ-ÖØ-Þ][a-zà-öø-ÿœæ]{2,}){1,3}\b"
)


def normaliser_manuellement(texte):
    return restaurer_apostrophes(texte)


def masquer_entites(texte):
    texte = normaliser_manuellement(texte)
    texte = MOTIF_SUITE_CAPITALISEE.sub(" ENTITE_NOMMEE ", texte)
    return MOTIF_NOMS_PROPRES.sub(" ENTITE_NOMMEE ", texte)


def extraire_mots_fonctionnels(texte):
    texte = normaliser_manuellement(texte)
    return " ".join(
        token.lower()
        for token in MOTIF_TOKEN_VARIANTE.findall(texte)
        if token.lower() in MOTS_FONCTIONNELS_FR
    )


def formes_lexicales(textes):
    formes = set()
    for texte in textes:
        texte = normaliser_manuellement(texte)
        for token in MOTIF_TOKEN_VARIANTE.findall(texte):
            forme = token.lower()
            if (
                len(forme) >= 2
                and forme.isalpha()
                and forme not in MOTS_FONCTIONNELS_FR
                and forme not in stop_words_noms_propres_set
            ):
                formes.add(forme)
    return sorted(formes)


def construire_lexique_lemmes(textes):
    """Lemmatise chaque type une fois avec le modèle français installé."""

    import spacy

    formes = formes_lexicales(textes)
    print(f"Formes lexicales uniques à lemmatiser : {len(formes)}")

    try:
        nlp = spacy.load(
            "fr_core_news_lg",
            exclude=["ner", "parser", "senter"],
        )
    except OSError as erreur:
        raise RuntimeError(
            "Le modèle spaCy fr_core_news_lg est requis pour la variante "
            "lexicale lemmatisée."
        ) from erreur

    lexique = {}
    documents = nlp.pipe(formes, batch_size=128, n_process=1)

    for forme, document in zip(formes, documents):
        if document and document[0].lemma_:
            lemme = document[0].lemma_.lower()
        else:
            lemme = forme
        lexique[forme] = lemme if lemme.isalpha() else forme

    del nlp
    gc.collect()
    return lexique


def extraire_contenu_lexical_lemmatise(texte, lexique_lemmes):
    texte = normaliser_manuellement(texte)
    lemmes = []

    for token in MOTIF_TOKEN_VARIANTE.findall(texte):
        forme = token.lower()
        if (
            len(forme) >= 2
            and forme.isalpha()
            and forme not in MOTS_FONCTIONNELS_FR
            and forme not in stop_words_noms_propres_set
        ):
            lemmes.append(lexique_lemmes.get(forme, forme))

    return " ".join(lemmes)


CONFIG_VARIANTES = {
    "texte_brut": {
        "stop_words": None,
        "token_pattern": r"(?u)\b\w\w+\b",
        "ngram_range": (1, 2),
        "max_df": 0.90,
    },
    "filtrage_manuel": {
        "stop_words": stop_words,
        "token_pattern": r"(?u)\b\w\w+\b",
        "ngram_range": (1, 2),
        "max_df": 0.90,
    },
    "entites_masquees": {
        "stop_words": None,
        "token_pattern": r"(?u)\b\w\w+\b",
        "ngram_range": (1, 2),
        "max_df": 0.90,
    },
    "mots_fonctionnels_seuls": {
        "stop_words": None,
        "token_pattern": r"(?u)\b\w+\b",
        "ngram_range": (1, 3),
        "max_df": 1.0,
    },
    "contenu_lexical_lemmatise": {
        "stop_words": None,
        "token_pattern": r"(?u)\b\w\w+\b",
        "ngram_range": (1, 2),
        "max_df": 0.90,
    },
}


def construire_variante(nom_variante, textes, lexique_lemmes):
    if nom_variante == "texte_brut":
        return np.asarray(list(textes), dtype=object)

    if nom_variante == "filtrage_manuel":
        fonction = normaliser_manuellement
    elif nom_variante == "entites_masquees":
        fonction = masquer_entites
    elif nom_variante == "mots_fonctionnels_seuls":
        fonction = extraire_mots_fonctionnels
    elif nom_variante == "contenu_lexical_lemmatise":
        fonction = lambda texte: extraire_contenu_lexical_lemmatise(
            texte,
            lexique_lemmes,
        )
    else:
        raise ValueError(f"Variante inconnue : {nom_variante}")

    return np.asarray([fonction(texte) for texte in textes], dtype=object)


# Le lexique est construit sur les formes observées, sans utiliser les
# labels. Il sert ensuite aux deux ensembles et à tous les plis.
textes_pour_lexique = pd.concat(
    [df_train["texte_brut"], df_developpement["texte_brut"]],
    ignore_index=True,
)
lexique_lemmes = construire_lexique_lemmes(textes_pour_lexique)

# Contrôle d'alignement : chaque variante doit produire une sortie par
# fenêtre, sans modifier les plis ni l'ordre des lignes.
for nom_variante in CONFIG_VARIANTES:
    exemple = construire_variante(
        nom_variante,
        df_train["texte_brut"].head(3),
        lexique_lemmes,
    )
    assert len(exemple) == 3

print("Cinq variantes prêtes sur les mêmes fenêtres.")


Formes lexicales uniques à lemmatiser : 81961
Cinq variantes prêtes sur les mêmes fenêtres.


## 7. Plis communs et poids égaux par œuvre

Les cinq variantes, les trois modèles et les deux baselines utilisent les
mêmes cinq plis internes. Une œuvre ne peut jamais apparaître à la fois
dans l'apprentissage et la validation d'un pli.

Chaque fenêtre reçoit un poids inverse du nombre de fenêtres de son œuvre.
La somme des poids est donc identique pour toutes les œuvres, pendant
l'apprentissage comme pendant le calcul des scores.


In [7]:
def poids_egaux_par_oeuvre(dataframe):
    effectifs = dataframe.groupby("source")["source"].transform("size")
    poids = 1.0 / effectifs.to_numpy(dtype=float)
    return poids / poids.mean()


y_train = df_train["label"].to_numpy()
groupes_train = df_train["source"].to_numpy()
poids_train = poids_egaux_par_oeuvre(df_train)
poids_developpement = poids_egaux_par_oeuvre(df_developpement)

validation_groupee = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=GRAINE_ALEATOIRE,
)

# Cette liste est créée une seule fois et réutilisée partout.
plis_communs = list(
    validation_groupee.split(
        X=np.zeros(len(df_train)),
        y=y_train,
        groups=groupes_train,
    )
)

sommes_par_oeuvre = (
    pd.DataFrame({"source": groupes_train, "poids": poids_train})
    .groupby("source")["poids"]
    .sum()
)
assert np.allclose(sommes_par_oeuvre, sommes_par_oeuvre.iloc[0])

controle_plis = []
for numero, (indices_apprentissage, indices_validation) in enumerate(
    plis_communs,
    start=1,
):
    oeuvres_apprentissage = set(groupes_train[indices_apprentissage])
    oeuvres_validation = set(groupes_train[indices_validation])
    communes = oeuvres_apprentissage & oeuvres_validation
    assert not communes
    controle_plis.append({
        "pli": numero,
        "œuvres_apprentissage": len(oeuvres_apprentissage),
        "œuvres_validation": len(oeuvres_validation),
        "intersection": len(communes),
    })

display(pd.DataFrame(controle_plis))


,pli,œuvres_apprentissage,œuvres_validation,intersection
0,1,40,9,0
1,2,38,11,0
2,3,39,10,0
3,4,40,9,0
4,5,39,10,0


## 8. Baselines et sélection groupée des paramètres

La métrique de sélection est le F1 macro pondéré : chaque œuvre contribue
au même poids total. Les paramètres sont choisis exclusivement par la
moyenne des cinq validations groupées internes.

Modèles comparés : régression logistique, SVM linéaire et Complement Naive
Bayes. Les deux baselines sont :

- prédiction de la classe majoritaire pondérée par œuvre ;
- régression logistique sur les seuls artefacts de surface (apostrophes,
  lignes et formes OCR), sans TF-IDF lexical.


In [8]:
GRILLES_PARAMETRES = {
    "Regression_logistique": {"C": [0.1, 1.0, 10.0]},
    "Linear_SVM": {"C": [0.1, 1.0, 10.0]},
    "Complement_NB": {"alpha": [0.1, 0.5, 1.0]},
}


def creer_vectoriseur(nom_variante):
    configuration = CONFIG_VARIANTES[nom_variante]
    return TfidfVectorizer(
        max_features=10000,
        min_df=2,
        sublinear_tf=True,
        stop_words=configuration["stop_words"],
        token_pattern=configuration["token_pattern"],
        ngram_range=configuration["ngram_range"],
        max_df=configuration["max_df"],
    )


def creer_modele(nom_modele, valeur):
    if nom_modele == "Regression_logistique":
        return LogisticRegression(
            C=valeur,
            max_iter=3000,
            solver="liblinear",
            random_state=GRAINE_ALEATOIRE,
        )
    if nom_modele == "Linear_SVM":
        return LinearSVC(
            C=valeur,
            max_iter=10000,
            random_state=GRAINE_ALEATOIRE,
        )
    if nom_modele == "Complement_NB":
        return ComplementNB(alpha=valeur)
    raise ValueError(f"Modèle inconnu : {nom_modele}")


def scores_ponderes(y_reel, y_predit, poids):
    return {
        "f1_macro": f1_score(
            y_reel,
            y_predit,
            average="macro",
            sample_weight=poids,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_reel,
            y_predit,
            sample_weight=poids,
        ),
    }


def classe_majoritaire_ponderee(labels, poids):
    sommes = pd.Series(poids).groupby(pd.Series(labels)).sum()
    return sommes.idxmax()


NOMS_ARTEFACTS = [
    "apostrophes_par_1000_caracteres",
    "longueur_moyenne_lignes",
    "ecart_type_longueur_lignes",
    "formes_ocr_par_1000_tokens",
    "caracteres_non_alphabetiques_par_1000",
]


def caracteristiques_artefacts(texte):
    nombre_caracteres = max(len(texte), 1)
    lignes = [ligne for ligne in texte.splitlines() if ligne.strip()]
    longueurs_lignes = [len(ligne) for ligne in lignes] or [0]
    tokens = MOTIF_TOKEN_FENETRE.findall(texte)
    nombre_tokens = max(len(tokens), 1)

    apostrophes = texte.count("'") + texte.count("’")
    formes_ocr = len(MOTIF_APOSTROPHES.findall(texte))
    non_alphabetiques = sum(
        not caractere.isalpha() and not caractere.isspace()
        for caractere in texte
    )

    return [
        1000 * apostrophes / nombre_caracteres,
        float(np.mean(longueurs_lignes)),
        float(np.std(longueurs_lignes)),
        1000 * formes_ocr / nombre_tokens,
        1000 * non_alphabetiques / nombre_caracteres,
    ]


def matrice_artefacts(textes):
    return np.asarray(
        [caracteristiques_artefacts(texte) for texte in textes],
        dtype=float,
    )


def resumer_selection(resultats_detail):
    resume = (
        resultats_detail
        .groupby(
            ["variante", "modele", "parametre", "valeur"],
            as_index=False,
        )
        .agg(
            f1_macro_cv=("f1_macro", "mean"),
            ecart_type_f1=("f1_macro", "std"),
            balanced_accuracy_cv=("balanced_accuracy", "mean"),
        )
    )

    return (
        resume
        .sort_values(
            ["variante", "modele", "f1_macro_cv"],
            ascending=[True, True, False],
        )
        .groupby(["variante", "modele"], as_index=False)
        .head(1)
        .sort_values("f1_macro_cv", ascending=False)
        .reset_index(drop=True)
    )


In [10]:
debut_selection = time.time()
resultats_cv_lignes = []

# Baseline 1 : majorité pondérée par œuvre.
for numero_pli, (indices_apprentissage, indices_validation) in enumerate(
    plis_communs,
    start=1,
):
    majorite = classe_majoritaire_ponderee(
        y_train[indices_apprentissage],
        poids_train[indices_apprentissage],
    )
    predictions = np.repeat(majorite, len(indices_validation))
    scores = scores_ponderes(
        y_train[indices_validation],
        predictions,
        poids_train[indices_validation],
    )
    resultats_cv_lignes.append({
        "variante": "aucune",
        "modele": "Baseline_majoritaire",
        "parametre": "aucun",
        "valeur": "classe_majoritaire",
        "pli": numero_pli,
        **scores,
    })

# Baseline 2 : artefacts uniquement.
X_artefacts_train = matrice_artefacts(df_train["texte_brut"])
for valeur_c in GRILLES_PARAMETRES["Regression_logistique"]["C"]:
    for numero_pli, (indices_apprentissage, indices_validation) in enumerate(
        plis_communs,
        start=1,
    ):
        standardiseur = StandardScaler()
        X_apprentissage = standardiseur.fit_transform(
            X_artefacts_train[indices_apprentissage]
        )
        X_validation = standardiseur.transform(
            X_artefacts_train[indices_validation]
        )
        modele = creer_modele("Regression_logistique", valeur_c)
        modele.fit(
            X_apprentissage,
            y_train[indices_apprentissage],
            sample_weight=poids_train[indices_apprentissage],
        )
        predictions = modele.predict(X_validation)
        scores = scores_ponderes(
            y_train[indices_validation],
            predictions,
            poids_train[indices_validation],
        )
        resultats_cv_lignes.append({
            "variante": "artefacts_seulement",
            "modele": "Baseline_artefacts",
            "parametre": "C",
            "valeur": valeur_c,
            "pli": numero_pli,
            **scores,
        })

# Les matrices TF-IDF sont apprises une fois par variante et par pli,
# puis réutilisées par tous les modèles et toutes les valeurs testées.
for nom_variante in CONFIG_VARIANTES:
    print(f"Validation de la variante : {nom_variante}")
    textes_variante = construire_variante(
        nom_variante,
        df_train["texte_brut"],
        lexique_lemmes,
    )

    for numero_pli, (indices_apprentissage, indices_validation) in enumerate(
        plis_communs,
        start=1,
    ):
        vectoriseur_pli = creer_vectoriseur(nom_variante)
        X_apprentissage = vectoriseur_pli.fit_transform(
            textes_variante[indices_apprentissage]
        )
        X_validation = vectoriseur_pli.transform(
            textes_variante[indices_validation]
        )

        for nom_modele, grille in GRILLES_PARAMETRES.items():
            parametre, valeurs = next(iter(grille.items()))
            for valeur in valeurs:
                modele = creer_modele(nom_modele, valeur)
                modele.fit(
                    X_apprentissage,
                    y_train[indices_apprentissage],
                    sample_weight=poids_train[indices_apprentissage],
                )
                predictions = modele.predict(X_validation)
                scores = scores_ponderes(
                    y_train[indices_validation],
                    predictions,
                    poids_train[indices_validation],
                )
                resultats_cv_lignes.append({
                    "variante": nom_variante,
                    "modele": nom_modele,
                    "parametre": parametre,
                    "valeur": valeur,
                    "pli": numero_pli,
                    **scores,
                })

    del textes_variante
    gc.collect()

resultats_cv_detail = pd.DataFrame(resultats_cv_lignes)
meilleurs_parametres_cv = resumer_selection(resultats_cv_detail)

print(
    "Temps de sélection groupée : "
    f"{(time.time() - debut_selection) / 60:.1f} minutes"
)
display(meilleurs_parametres_cv)


Validation de la variante : texte_brut
Validation de la variante : filtrage_manuel
Validation de la variante : entites_masquees
Validation de la variante : mots_fonctionnels_seuls
Validation de la variante : contenu_lexical_lemmatise
Temps de sélection groupée : 11.1 minutes


,variante,modele,parametre,valeur,f1_macro_cv,ecart_type_f1,balanced_accuracy_cv
0,texte_brut,Linear_SVM,C,10.0,0.985932,0.005807,0.988453
1,entites_masquees,Linear_SVM,C,1.0,0.982863,0.004725,0.982097
2,filtrage_manuel,Linear_SVM,C,1.0,0.981798,0.005570,0.981017
3,texte_brut,Regression_logistique,C,10.0,0.981308,0.010152,0.983035
4,entites_masquees,Regression_logistique,C,10.0,0.980722,0.005225,0.976844
5,filtrage_manuel,Regression_logistique,C,10.0,0.978941,0.005734,0.974947
6,contenu_lexical_lemmatise,Linear_SVM,C,10.0,0.973740,0.008397,0.974748
7,contenu_lexical_lemmatise,Regression_logistique,C,10.0,0.969986,0.011308,0.966398
8,artefacts_seulement,Baseline_artefacts,C,1.0,0.936564,0.049110,0.945028
9,mots_fonctionnels_seuls,Regression_logistique,C,10.0,0.919839,0.011693,0.921360


## 9. Évaluation exploratoire sur le jeu de développement

Chaque configuration est réentraînée avec le paramètre choisi uniquement
par validation groupée. Le développement sert à diagnostiquer la
généralisation ; il ne doit pas être présenté comme un test final et ses
résultats ne doivent pas conduire à modifier les paramètres.

Deux scores sont rapportés : le F1 macro des fenêtres avec poids égaux par
œuvre, et le F1 macro au niveau des œuvres après vote majoritaire des
fenêtres.


In [11]:
def f1_macro_par_oeuvre(dataframe, predictions):
    evaluation = dataframe[["source", "label"]].copy()
    evaluation["prediction"] = predictions

    par_oeuvre = (
        evaluation
        .groupby("source")
        .agg(
            label_reel=("label", "first"),
            prediction=(
                "prediction",
                lambda serie: serie.value_counts().sort_index().idxmax(),
            ),
        )
        .reset_index()
    )

    return f1_score(
        par_oeuvre["label_reel"],
        par_oeuvre["prediction"],
        average="macro",
    )


def ajouter_resultat_developpement(
    lignes,
    variante,
    modele,
    parametre,
    valeur,
    predictions,
):
    scores = scores_ponderes(
        df_developpement["label"].to_numpy(),
        predictions,
        poids_developpement,
    )
    lignes.append({
        "variante": variante,
        "modele": modele,
        "parametre": parametre,
        "valeur": valeur,
        "f1_macro_fenetres_pondere": scores["f1_macro"],
        "balanced_accuracy_ponderee": scores["balanced_accuracy"],
        "f1_macro_oeuvres": f1_macro_par_oeuvre(
            df_developpement,
            predictions,
        ),
    })


resultats_developpement_lignes = []
modeles_developpement = {}
y_developpement = df_developpement["label"].to_numpy()

# Majorité : aucun paramètre n'est choisi sur le développement.
majorite_train = classe_majoritaire_ponderee(y_train, poids_train)
predictions_majorite = np.repeat(
    majorite_train,
    len(df_developpement),
)
ajouter_resultat_developpement(
    resultats_developpement_lignes,
    "aucune",
    "Baseline_majoritaire",
    "aucun",
    majorite_train,
    predictions_majorite,
)

# Artefacts : C provient exclusivement de la validation groupée.
ligne_artefacts = meilleurs_parametres_cv.query(
    "modele == 'Baseline_artefacts'"
).iloc[0]
meilleur_c_artefacts = float(ligne_artefacts["valeur"])
standardiseur_artefacts = StandardScaler()
X_artefacts_train_final = standardiseur_artefacts.fit_transform(
    X_artefacts_train
)
X_artefacts_developpement = standardiseur_artefacts.transform(
    matrice_artefacts(df_developpement["texte_brut"])
)
modele_artefacts = creer_modele(
    "Regression_logistique",
    meilleur_c_artefacts,
)
modele_artefacts.fit(
    X_artefacts_train_final,
    y_train,
    sample_weight=poids_train,
)
predictions_artefacts = modele_artefacts.predict(
    X_artefacts_developpement
)
ajouter_resultat_developpement(
    resultats_developpement_lignes,
    "artefacts_seulement",
    "Baseline_artefacts",
    "C",
    meilleur_c_artefacts,
    predictions_artefacts,
)
modeles_developpement[("artefacts_seulement", "Baseline_artefacts")] = {
    "standardiseur": standardiseur_artefacts,
    "modele": modele_artefacts,
    "variables": NOMS_ARTEFACTS,
}

# Modèles textuels : un TF-IDF final par variante, paramètres issus de CV.
for nom_variante in CONFIG_VARIANTES:
    print(f"Évaluation développement : {nom_variante}")
    textes_train = construire_variante(
        nom_variante,
        df_train["texte_brut"],
        lexique_lemmes,
    )
    textes_developpement = construire_variante(
        nom_variante,
        df_developpement["texte_brut"],
        lexique_lemmes,
    )

    vectoriseur_final = creer_vectoriseur(nom_variante)
    X_train_final = vectoriseur_final.fit_transform(textes_train)
    X_developpement_final = vectoriseur_final.transform(
        textes_developpement
    )

    lignes_variante = meilleurs_parametres_cv.query(
        "variante == @nom_variante"
    )
    for _, ligne in lignes_variante.iterrows():
        nom_modele = ligne["modele"]
        valeur = float(ligne["valeur"])
        modele = creer_modele(nom_modele, valeur)
        modele.fit(
            X_train_final,
            y_train,
            sample_weight=poids_train,
        )
        predictions = modele.predict(X_developpement_final)

        ajouter_resultat_developpement(
            resultats_developpement_lignes,
            nom_variante,
            nom_modele,
            ligne["parametre"],
            valeur,
            predictions,
        )
        modeles_developpement[(nom_variante, nom_modele)] = {
            "vectoriseur": vectoriseur_final,
            "modele": modele,
        }

    del textes_train, textes_developpement
    del X_train_final, X_developpement_final
    gc.collect()

resultats_developpement = (
    pd.DataFrame(resultats_developpement_lignes)
    .sort_values("f1_macro_oeuvres", ascending=False)
    .reset_index(drop=True)
)

display(resultats_developpement)


Évaluation développement : texte_brut
Évaluation développement : filtrage_manuel
Évaluation développement : entites_masquees
Évaluation développement : mots_fonctionnels_seuls
Évaluation développement : contenu_lexical_lemmatise


,variante,modele,parametre,valeur,f1_macro_fenetres_pondere,balanced_accuracy_ponderee,f1_macro_oeuvres
0,entites_masquees,Linear_SVM,C,1.0,0.977560,0.979489,1.000000
1,artefacts_seulement,Baseline_artefacts,C,1.0,0.967964,0.964381,1.000000
2,contenu_lexical_lemmatise,Regression_logistique,C,10.0,0.963342,0.959691,1.000000
3,contenu_lexical_lemmatise,Linear_SVM,C,10.0,0.966977,0.967172,1.000000
4,mots_fonctionnels_seuls,Linear_SVM,C,1.0,0.916179,0.922036,1.000000
5,mots_fonctionnels_seuls,Regression_logistique,C,10.0,0.922628,0.924511,1.000000
6,entites_masquees,Regression_logistique,C,10.0,0.973360,0.974694,1.000000
7,contenu_lexical_lemmatise,Complement_NB,alpha,0.1,0.919051,0.930646,1.000000
8,filtrage_manuel,Regression_logistique,C,10.0,0.975645,0.976802,1.000000
9,filtrage_manuel,Linear_SVM,C,1.0,0.977118,0.979610,1.000000


## 10. Statut des résultats

- Les cinq variantes utilisent les mêmes fenêtres, le même jeu de
  développement et les mêmes plis internes.
- La sélection des paramètres ne consulte jamais le développement.
- Toutes les métriques de fenêtre donnent le même poids total à chaque
  œuvre ; un score au niveau des œuvres est également fourni.
- Le développement actuel reste exploratoire. Aucun résultat de cette
  section ne constitue encore une mesure finale.
- Un futur test externe devra être chargé dans un autre dataframe et ne
  devra être évalué qu'après gel complet du protocole.

L'étiquetage par nom de fichier est conservé ici pour respecter le périmètre
de la modification demandée. Il devra encore être remplacé par le manifeste
explicite des œuvres avant le dépôt universitaire.


## Annexe — ancienne analyse exploratoire (non exécutable)

Les cellules suivantes sont conservées pour assurer la traçabilité de la version précédente. Elles ne font plus partie du protocole.
